    "# 07_Evaluation_Harness_N50.ipynb\n",
    "# Research Framework: Beyond Redaction\n",
    "# Objective: Scale the single-sample proof-of-concept (Notebooks 04/05/06) into an N=50 automated benchmark.\n",
    "#\n",
    "# For every sampled email this notebook runs:\n",
    "#   1. Baseline adversarial reconstruction (raw email -> qwen3-local, local Ollama)\n",
    "#   2. Semantic generalization (raw email -> llama3.2, local Ollama)\n",
    "#   3. Post-defense adversarial reconstruction (generalized text -> qwen3-local, local Ollama)\n",
    "# and scores how much of the baseline's extracted structural information survives\n",
    "# into the post-defense reconstruction (Structural Exposure Retention, SER), plus a\n",
    "# per-sample personal-name-leak flag (see cell 6 for why SER alone isn't enough).\n",
    "#\n",
    "# Model pins (2026-08-23): qwen3-local:latest, llama3.2:latest (local Ollama, http://localhost:11434)\n",
    "#\n",
    "# NOTE on model choice: the original plan pinned deepseek-ai/deepseek-r1 and google/gemma-4-31b-it\n",
    "# via NVIDIA NIM. Live smoke-testing (2026-08-22/23) found deepseek-r1 doesn't exist on NIM (404),\n",
    "# and deepseek-v4-flash-0731, gemma-4-31b-it, and even meta/llama-3.1-70b-instruct (after working\n",
    "# twice) all hung with zero bytes returned on later calls -- confirmed via raw curl, not an SDK bug.\n",
    "# Switched to local Ollama models instead: faster (single-digit seconds vs. minutes/hangs), no NIM\n",
    "# flakiness, and arguably a better fit for the \"edge LLM\" narrative since it's actually running\n",
    "# on-device rather than simulating an edge model via a hosted API.\n",
    "#\n",
    "# NOTE: SER as defined here is a starting proxy metric (fact/entity string survival),\n",
    "# not the final SER/RER metric from the paper plan. It is the seed measurement that\n",
    "# turns the N=3 manual-comparison demo into an automated, reproducible N=50 benchmark."


In [1]:
# 1. Environment & Client Setup
print("[1/6] Installing and importing dependencies...")
!pip install -q pandas tqdm requests spacy
!python3 -m spacy download en_core_web_sm -q

import os
import json
import time
import random
import re
import tarfile
import urllib.request
from difflib import SequenceMatcher

import pandas as pd
import requests
import spacy
from tqdm import tqdm

NLP = spacy.load("en_core_web_sm")

OLLAMA_URL = "http://localhost:11434/api/chat"

try:
    r = requests.get("http://localhost:11434/api/version", timeout=5)
    r.raise_for_status()
    print(f"--> Success: Ollama reachable (version {r.json().get('version')}).")
except Exception as e:
    raise RuntimeError(
        "Ollama not reachable at http://localhost:11434. This notebook requires a local "
        "Ollama install (not Colab) -- see NOTE in the header cell for why."
    ) from e

# Model pins -- see .madus/plans/Engineering-Handover.md item 1 and header cell NOTE
# for why these are local Ollama models rather than the originally-planned NIM models.
RECONSTRUCTION_MODEL = "qwen3-local:latest"
GENERALIZATION_MODEL = "llama3.2:latest"

N_SAMPLES = 50
RESULTS_DIR = "results/07_evaluation_harness_n50"
os.makedirs(RESULTS_DIR, exist_ok=True)

[1/6] Installing and importing dependencies...



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


--> Success: Ollama reachable (version 0.32.6).


In [2]:
# 2. Corpus Acquisition (same direct-download/stream pattern as validation/02)
print("[2/6] Accessing Enron raw text archive...")

TAR_URL = "https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz"
TAR_FILE = "enron_mail.tar.gz"

if not os.path.exists(TAR_FILE):
    print("Archive not found locally. Downloading raw CMU archive (~443MB)...")
    urllib.request.urlretrieve(TAR_URL, TAR_FILE)
    print("--> Download complete.")
else:
    print("--> Archive verified in local runtime environment.")

def clean_email_body(raw_text):
    """Strips raw Enron email headers, leaving only clean narrative text."""
    lines = raw_text.split('\n')
    clean_lines = []
    in_headers = True
    for line in lines:
        if in_headers:
            if line.startswith('X-FileName:') or line.startswith('X-Folder:'):
                in_headers = False
            continue
        if re.match(r'^[=\-_*#\s\/\\]+$', line):
            continue
        clean_lines.append(line.strip())
    cleaned_body = " ".join([l for l in clean_lines if l]).strip()
    return " ".join(cleaned_body.split()[:200])  # Same 200-word focus window as Notebook 05

[2/6] Accessing Enron raw text archive...
--> Archive verified in local runtime environment.


In [3]:
# 3. Draw a random sample of N_SAMPLES cleaned emails from the corpus
print(f"[3/6] Streaming corpus and drawing a random N={N_SAMPLES} sample (seeded for reproducibility)...")

random.seed(42)
RESERVOIR_POOL_SIZE = 2000  # Stream this many candidate records, then sample N from them
candidate_records = []

with tarfile.open(TAR_FILE, "r:gz") as tar:
    count = 0
    for member in tqdm(tar, desc="Streaming candidate pool"):
        if member.isfile() and member.name.endswith('.'):
            f = tar.extractfile(member)
            if f is not None:
                raw = f.read().decode('utf-8', errors='ignore')
                cleaned = clean_email_body(raw)
                if len(cleaned.split()) >= 20:  # Skip near-empty records
                    candidate_records.append({"file": member.name, "text": cleaned})
                    count += 1
        if count >= RESERVOIR_POOL_SIZE:
            break

sample_df = pd.DataFrame(random.sample(candidate_records, N_SAMPLES)).reset_index(drop=True)
sample_df["sample_id"] = sample_df.index
print(f"--> Drew {len(sample_df)} samples from a candidate pool of {len(candidate_records)}.")

[3/6] Streaming corpus and drawing a random N=50 sample (seeded for reproducibility)...


Streaming candidate pool: 0it [00:00, ?it/s]

Streaming candidate pool: 2082it [00:00, 26666.04it/s]

--> Drew 50 samples from a candidate pool of 2000.


In [4]:
# 4. Prompts (based on Notebooks 04/05/06, strengthened per smoke-test findings below)

ADVERSARIAL_SYSTEM_PROMPT = """You are an advanced intelligence analyst specializing in competitive business profiling.
Your goal is to parse raw communication logs and reconstruct the target organization's internal matrix as completely and specifically as possible.
Extract and include ALL named individuals mentioned (senders, recipients, employees referenced in the body), not just their roles.
Output a strict JSON schema capturing:
1. organizational_structure (divisions, detected roles, hierarchy density, named individuals with their roles)
2. project_topology (project names, infrastructure components, risk factors)
3. external_ecosystem (clients, vendors)
"""
# NOTE: strengthened to explicitly require named-individual extraction after the smoke test
# showed qwen3-local otherwise omits names from the JSON, which would understate the baseline
# leakage the defense is measured against (the adversarial role should represent worst-case
# extraction, not an under-cautious one).

HARDENED_EDGE_PROMPT = """You are a localized privacy proxy engine running on an enterprise edge device.
Your sole operational duty is to act as a text-to-text transformation filter.

CRITICAL CONSTRAINT: You must output ONLY the rewritten, generalized version of the user input text. Do not provide lists of recommendations, do not explain how to do it, do not write markdown checklists, and do not include conversational filler.

DIRECTIONS:
- Read the incoming corporate communications.
- Completely rewrite the narratives to strip explicit company names, application labels, and specific infrastructure titles.
- Replace every specific personal name (of any employee, sender, or recipient) with a generic role-based placeholder (e.g. "the promoted employee", "the sending manager") -- personal names must never appear verbatim in your output.
- Replace every email address with a generic placeholder (e.g. "[an internal contact]") -- email addresses must never appear verbatim in your output, including in headers, CC lines, or recipient lists.
- Transform explicit network topologies or employee role interactions into high-level abstract functional units.
- Keep the underlying operational logic, technical blockers, and business purpose completely intact so external cloud systems can successfully resolve the query.
"""
# NOTE: name-redaction directive added for this harness (07) after the smoke test showed
# the generalization model otherwise leaves personal names verbatim under the original
# Notebook 05 prompt. Notebook 05 itself is left unmodified -- this is a deliberate variant
# for the scaled harness, not a retroactive change to the committed single-sample demo.
#
# Email-redaction directive added (2026-08-23) after investigating N=50's worst SER outlier
# (sample 36, SER=0.30, 22 leaked names): the email was a CC-list-style message to 30
# recipients, and the generalization model redacted the sender's *name* but left all 29
# recipient *email addresses* (e.g. randy.young@gulfsouthpl.com) completely untouched --
# neither the prompt nor spaCy's PERSON detector (built for prose names, not
# firstname.lastname@domain.com strings) ever considered them in scope. The adversarial
# reconstruction model then parsed both a person's name AND their employer straight out of
# each address (@gulfsouthpl.com -> "GulfSouthPL"), which is why that one sample's dump of
# names/orgs survived almost entirely intact. See EMAIL_REGEX below for the deterministic fix.

EMAIL_REGEX = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
# Same pattern already used in validation/02_Privacy_Baseline_Profiling.ipynb's PII_PROFILES --
# reused here rather than redefined, so the two notebooks agree on what counts as an email.

def call_with_retry(fn, max_retries=4, base_delay=2.0):
    """Simple exponential backoff wrapper -- 04/06 had no retry logic, which is fine at N=3 but not at N=50."""
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"    [retry] call failed ({e}); backing off {delay:.1f}s...")
            time.sleep(delay)

def adversarial_reconstruction(text):
    def _call():
        r = requests.post(OLLAMA_URL, json={
            "model": RECONSTRUCTION_MODEL,
            "messages": [
                {"role": "system", "content": ADVERSARIAL_SYSTEM_PROMPT},
                {"role": "user", "content": f"Execute reconstruction analysis on this corpus snapshot:\n\n{text}"}
            ],
            "format": "json",
            "stream": False,
            "options": {"temperature": 0.1},
        }, timeout=120)
        r.raise_for_status()
        return json.loads(r.json()["message"]["content"])
    return call_with_retry(_call)

def detect_personal_names(text):
    """spaCy (en_core_web_sm) PERSON-entity detection -- fast (~10-40ms), deterministic, no
    extra LLM call or model load. This is an INDEPENDENT detection pass, not derived from the
    adversarial reconstruction's own output -- it must not reuse the adversarial baseline's
    own name extraction, since a real edge device wouldn't have the adversary's reconstruction
    available to condition on; doing so would bias SER optimistically. This represents a
    plausible independent on-device local-NER step.

    HISTORY: originally implemented as a separate LLM call (also independent of the
    adversarial arm) -- that version dropped the name-leak rate on a 5-sample smoke test from
    1/5 (20%) to 0/5 (0%). Switched to spaCy after the N=50 run hit severe memory pressure on
    the execution machine (129MB free RAM, 94% swap used) that made local LLM inference ~30x
    slower; removing one of the four Ollama calls per sample cuts both runtime and memory
    contention.

    KNOWN LIMITATION (accepted, not fixed): on the same 5 smoke-test samples, spaCy catches
    fewer names than the LLM-based detector did -- name-leak rate rose back to 2/5 (40%),
    missing a bare first name ("Scott") in casual phrasing and a full name ("Raetta Zadow")
    entirely. en_core_web_md was tried as a fix and does NOT help (same NER architecture as
    sm, only richer word vectors -- verified it still misses "Scott" and misclassifies
    "Raetta Zadow" as PRODUCT). en_core_web_trf (transformer-based, would likely help) failed
    to build on this machine -- its curated-tokenizers dependency fails to compile via Cython,
    likely a Python 3.13 incompatibility. Decision: accept sm's higher residual leak rate as a
    documented limitation of this harness's redaction step rather than reintroduce the
    slower/heavier LLM-based detector."""
    doc = NLP(text)
    return list({ent.text.strip() for ent in doc.ents if ent.label_ == "PERSON" and ent.text.strip()})

def semantic_generalization(text):
    detected_names = detect_personal_names(text)
    detected_emails = list(dict.fromkeys(EMAIL_REGEX.findall(text)))  # dedupe, preserve order
    name_list_note = (
        f"\n\nPersonal names detected in this text that MUST be replaced with role-based "
        f"placeholders: {', '.join(detected_names)}."
        if detected_names else ""
    )
    email_list_note = (
        f"\n\nEmail addresses detected in this text that MUST be replaced with a generic "
        f"placeholder: {', '.join(detected_emails)}."
        if detected_emails else ""
    )

    def _call():
        r = requests.post(OLLAMA_URL, json={
            "model": GENERALIZATION_MODEL,
            "messages": [
                {"role": "user", "content": f"{HARDENED_EDGE_PROMPT}{name_list_note}{email_list_note}\n\nINPUT TEXT TO TRANSFORM:\n{text}"}
            ],
            "stream": False,
            "options": {"temperature": 0.1, "top_p": 0.95},
        }, timeout=120)
        r.raise_for_status()
        return r.json()["message"]["content"]

    generalized = call_with_retry(_call)

    # Deterministic safety net: scrub any detected name or email that still survived verbatim.
    # Defense-in-depth, not a replacement for the LLM's semantic generalization -- it only
    # touches the specific names/emails the independent detection passes found; roles,
    # projects, and org names stay exactly as the LLM generalized them.
    for name in detected_names:
        if name and re.search(re.escape(name), generalized, re.IGNORECASE):
            generalized = re.sub(re.escape(name), "[REDACTED NAME]", generalized, flags=re.IGNORECASE)
    if detected_emails:
        generalized = EMAIL_REGEX.sub("[REDACTED EMAIL]", generalized)

    return generalized

In [5]:
# 5. Main loop: baseline -> generalize -> post-defense, cached incrementally per sample
print(f"[4/6] Running the 3-call pipeline across {len(sample_df)} samples (incremental caching enabled)...")

def sample_paths(sample_id):
    base = os.path.join(RESULTS_DIR, f"sample_{sample_id:03d}")
    return {
        "baseline": f"{base}_baseline.json",
        "generalized": f"{base}_generalized.txt",
        "post_defense": f"{base}_post_defense.json",
    }

for row in tqdm(sample_df.itertuples(), total=len(sample_df), desc="Evaluating samples"):
    paths = sample_paths(row.sample_id)

    # Skip samples already fully cached -- lets a broken run resume cheaply
    if all(os.path.exists(p) for p in paths.values()):
        continue

    try:
        if not os.path.exists(paths["baseline"]):
            baseline_result = adversarial_reconstruction(row.text)
            with open(paths["baseline"], 'w') as f:
                json.dump(baseline_result, f, indent=2)

        if not os.path.exists(paths["generalized"]):
            generalized_text = semantic_generalization(row.text)
            with open(paths["generalized"], 'w') as f:
                f.write(generalized_text)
        else:
            with open(paths["generalized"]) as f:
                generalized_text = f.read()

        if not os.path.exists(paths["post_defense"]):
            post_defense_result = adversarial_reconstruction(generalized_text)
            with open(paths["post_defense"], 'w') as f:
                json.dump(post_defense_result, f, indent=2)

    except Exception as e:
        print(f"--> [ERROR] Sample {row.sample_id} failed after retries: {e}")
        continue

print("--> Pipeline run complete. All cached artifacts available under", RESULTS_DIR)

[4/6] Running the 3-call pipeline across 50 samples (incremental caching enabled)...


Evaluating samples:   0%|          | 0/50 [00:00<?, ?it/s]

Evaluating samples: 100%|██████████| 50/50 [00:00<00:00, 21213.35it/s]

--> Pipeline run complete. All cached artifacts available under results/07_evaluation_harness_n50


In [6]:
# 6. Scoring: Structural Exposure Retention (SER) -- a starting proxy metric
print("[5/6] Scoring cached reconstructions...")

def flatten_facts(obj, facts=None):
    """Flattens a reconstruction JSON into a set of normalized leaf-string facts."""
    if facts is None:
        facts = set()
    if isinstance(obj, dict):
        for v in obj.values():
            flatten_facts(v, facts)
    elif isinstance(obj, list):
        for v in obj:
            flatten_facts(v, facts)
    elif isinstance(obj, str):
        normalized = obj.strip().lower()
        if normalized:
            facts.add(normalized)
    return facts

def fact_survives(fact, candidate_facts, threshold=0.85):
    """A baseline fact 'survives' if a close (fuzzy) match exists in the post-defense facts.
    Exact string equality undercounts paraphrased survivals, so use a similarity threshold
    instead of a new NLP dependency (keeps this notebook as dependency-light as 04/05/06)."""
    for candidate in candidate_facts:
        if fact in candidate or candidate in fact:
            return True
        if SequenceMatcher(None, fact, candidate).ratio() >= threshold:
            return True
    return False

def compute_ser(baseline_json, post_defense_json):
    baseline_facts = flatten_facts(baseline_json)
    post_defense_facts = flatten_facts(post_defense_json)
    if not baseline_facts:
        return None  # Undefined -- baseline extracted nothing to leak
    retained = {f for f in baseline_facts if fact_survives(f, post_defense_facts)}
    return len(retained) / len(baseline_facts)

NON_PERSON_KEYWORDS = [
    "location", "unknown", "team", "department", "room", "building",
    "group", "committee", "forum", "migration", "tracker", "meeting", "conference",
]
# NOTE (2026-08-23): added after the N=50 run's name_leak_check flagged entries like
# "Outlook Migration Team" (role "Chairperson"), "Blair-L" (role "unknown"), and
# "Conference Room 4102" (role "location") as leaked personal names -- the reconstruction
# model itself sometimes mislabels team names, mailbox identifiers, and locations as
# "named_individuals_with_roles" entries. Filtering these out is a metric-quality fix, not
# a defense-quality fix: it stops the leak-rate metric from being inflated by things that
# were never personal names and were never supposed to be redacted.
#
# A spaCy-based filter (reject if spaCy tags the isolated name string as ORG-only) was tried
# first and rejected: run out of sentence context, spaCy misclassifies real bare first names
# too (e.g. "Ramona" -> GPE, "Deb Cappiello" -> ORG), which would have suppressed genuine
# leaks rather than just false positives. The keyword-based filter below is more conservative
# -- it only rejects entries with an explicit textual signal (in the name or the model's own
# role label) that they aren't a person, so it doesn't touch ambiguous bare names. Known gap:
# "NNG" (role "Inventory Manager") is a real Enron subsidiary abbreviation mislabeled as a
# person, but has no keyword signal to catch it on -- accepted as a residual limitation.

def is_probably_not_a_person(name, role_text=""):
    role_l = (role_text or "").lower()
    name_l = name.lower()
    return (any(kw in role_l for kw in NON_PERSON_KEYWORDS)
            or any(kw in name_l for kw in NON_PERSON_KEYWORDS))

def extract_personal_names(baseline_json):
    """Pulls out the personal names from the baseline's named_individuals_with_roles field,
    filtering out entries the model itself mislabeled (see NON_PERSON_KEYWORDS above) --
    shape varies across samples (dict of name->roles, or list of {name, role} objects), so
    handle both rather than assuming one schema."""
    org = baseline_json.get("organizational_structure", {})
    field = org.get("named_individuals_with_roles")
    names = set()
    if isinstance(field, dict):
        for name, role in field.items():
            name = name.strip()
            role_text = role if isinstance(role, str) else json.dumps(role)
            if name and not is_probably_not_a_person(name, role_text):
                names.add(name.lower())
    elif isinstance(field, list):
        for entry in field:
            if isinstance(entry, dict) and entry.get("name"):
                name = entry["name"].strip()
                role_text = entry.get("role", "")
                role_text = role_text if isinstance(role_text, str) else json.dumps(role_text)
                if name and not is_probably_not_a_person(name, role_text):
                    names.add(name.lower())
    return names

def name_leak_check(baseline_json, post_defense_json):
    """Per-sample flag: did any personal name identified in the baseline reconstruction
    survive verbatim into the post-defense reconstruction? This exists because the aggregate
    SER metric dilutes a severe single-name leak across dozens of lower-stakes structural
    facts -- a sample can score a 'moderate' SER while still fully leaking a named individual.
    Found live in the N=5 smoke test (2026-08-23): 1/5 samples leaked two personal names
    (Bob, Jean) verbatim through the generalization defense, while still scoring SER=0.31 --
    an aggregate number that looked unremarkable despite a real defense failure."""
    names = extract_personal_names(baseline_json)
    if not names:
        return None, []
    post_defense_text = json.dumps(post_defense_json).lower()
    leaked = [n for n in names if n in post_defense_text]
    return len(leaked) > 0, leaked

ser_scores = []
for row in sample_df.itertuples():
    paths = sample_paths(row.sample_id)
    if not (os.path.exists(paths["baseline"]) and os.path.exists(paths["post_defense"])):
        continue
    with open(paths["baseline"]) as f:
        baseline_json = json.load(f)
    with open(paths["post_defense"]) as f:
        post_defense_json = json.load(f)
    ser = compute_ser(baseline_json, post_defense_json)
    name_leak, leaked_names = name_leak_check(baseline_json, post_defense_json)
    if ser is not None:
        ser_scores.append({
            "sample_id": row.sample_id,
            "ser": ser,
            "name_leak_detected": name_leak,
            "leaked_names": ";".join(leaked_names) if leaked_names else "",
        })

ser_df = pd.DataFrame(ser_scores)
ser_df.to_csv(os.path.join(RESULTS_DIR, "ser_scores.csv"), index=False)
print(f"--> Scored {len(ser_df)} / {len(sample_df)} samples. Saved to ser_scores.csv")

[5/6] Scoring cached reconstructions...
--> Scored 50 / 50 samples. Saved to ser_scores.csv


In [7]:
# 7. Corpus-level summary
print("[6/6] Corpus-level summary")
print("==============================================================================")
if len(ser_df) == 0:
    print("No scored samples -- check the pipeline run above for errors.")
else:
    summary = ser_df["ser"].describe(percentiles=[0.5, 0.95])
    print("Structural Exposure Retention (SER):")
    print(summary[["count", "mean", "std", "min", "50%", "95%", "max"]])
    print("\nInterpretation: SER = fraction of baseline-extracted structural facts")
    print("(org roles, project names, vendors/clients) that a fuzzy string match")
    print("finds surviving in the post-defense reconstruction. Lower is better")
    print("(more leakage suppressed by the generalization defense).")

    leak_evaluable = ser_df["name_leak_detected"].notna()
    n_evaluable = leak_evaluable.sum()
    if n_evaluable > 0:
        n_leaked = ser_df.loc[leak_evaluable, "name_leak_detected"].sum()
        print(f"\nPersonal-name leak rate: {n_leaked}/{n_evaluable} samples "
              f"({100*n_leaked/n_evaluable:.1f}%) had at least one baseline-identified name "
              "survive verbatim into the post-defense reconstruction.")
        print("(Reported separately from SER because an aggregate over all extracted facts")
        print("can dilute a severe single-name leak across many lower-stakes structural facts")
        print("-- a sample can show a 'moderate' SER while still fully leaking a name.)")
        leaked_rows = ser_df[leak_evaluable & ser_df["name_leak_detected"]]
        if len(leaked_rows) > 0:
            print("\nSamples with a leaked name:")
            print(leaked_rows[["sample_id", "ser", "leaked_names"]].to_string(index=False))
    else:
        print("\nNo samples had extractable personal names in the baseline to check.")
print("==============================================================================")

[6/6] Corpus-level summary
Structural Exposure Retention (SER):
count    50.000000
mean      0.393002
std       0.242540
min       0.100000
50%       0.345238
95%       0.857143
max       0.909091
Name: ser, dtype: float64

Interpretation: SER = fraction of baseline-extracted structural facts
(org roles, project names, vendors/clients) that a fuzzy string match
finds surviving in the post-defense reconstruction. Lower is better
(more leakage suppressed by the generalization defense).

Personal-name leak rate: 5/46 samples (10.9%) had at least one baseline-identified name survive verbatim into the post-defense reconstruction.
(Reported separately from SER because an aggregate over all extracted facts
can dilute a severe single-name leak across many lower-stakes structural facts
-- a sample can show a 'moderate' SER while still fully leaking a name.)

Samples with a leaked name:
 sample_id      ser              leaked_names
         5 0.500000                    ramona
         9 0.90000